# Phase 3-4 — L1 边缘模型训练与部署
# L1 Edge Model Training & TFLite Deployment

**EchoGlove Edge-AI Data Glove V3**

本文档覆盖：
1. 数据集加载与预处理
2. 1D-CNN + Temporal Attention 模型架构详解
3. Multi-Scale TCN 模型架构详解
4. 训练循环与超参数调优
5. 模型评估（混淆矩阵、逐类精度）
6. QAT 量化感知训练 → TFLite INT8 导出
7. 模型大小与推理延迟基准测试
8. ESP32-S3 部署（C 头文件生成）

---

## 双轨模型策略

| 模型 | 参数量 | 输入形状 | 适用场景 |
|------|--------|----------|----------|
| `CNN+Attention` | ~34K | `(B, 21)` 单帧 | 实时逐帧推理 |
| `MS-TCN` | ~12K | `(B, T, 21)` 窗口 | 时序模式识别 |

两者共享相同的特征管道和分类头，可灵活切换。

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import json
import os
import time
from pathlib import Path
from typing import Tuple, Dict, Any, List

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

PROJECT_ROOT = Path('.').resolve()
while not (PROJECT_ROOT / 'CLAUDE.md').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

## 1. 数据集加载

In [ ]:
# 常量
FEATURE_COUNT = 21
WINDOW_SIZE = 30
NUM_CLASSES = 20  # 合成数据集使用 20 类
FEATURE_NAMES = []
for i in range(5):
    for axis in ['x', 'y', 'z']:
        FEATURE_NAMES.append(f'hall_{i}_{axis}')
FEATURE_NAMES += ['euler_roll', 'euler_pitch', 'euler_yaw']
FEATURE_NAMES += ['gyro_x', 'gyro_y', 'gyro_z']

# 加载数据集
data_path = PROJECT_ROOT / 'data' / 'processed' / 'synthetic_dataset.npz'

if data_path.exists():
    data = np.load(data_path, allow_pickle=True)
    X_train = data['X_train']
    y_train = data['y_train']
    X_val = data['X_val']
    y_val = data['y_val']
    NUM_CLASSES = int(data['num_classes'])
    print(f'Loaded from {data_path}')
else:
    print(f'Dataset not found at {data_path}')
    print('Generating synthetic dataset inline...')
    # 复用 Notebook 01 的生成函数（简化版）
    np.random.seed(42)
    N = 2000
    X_train = np.random.randn(N, WINDOW_SIZE, FEATURE_COUNT).astype(np.float32) * 0.3 + 0.5
    y_train = np.random.randint(0, NUM_CLASSES, N)
    X_val = np.random.randn(500, WINDOW_SIZE, FEATURE_COUNT).astype(np.float32) * 0.3 + 0.5
    y_val = np.random.randint(0, NUM_CLASSES, 500)

print(f'Train: X={X_train.shape}, y={y_train.shape}')
print(f'Val:   X={X_val.shape}, y={y_val.shape}')
print(f'Classes: {NUM_CLASSES}')
print(f'X range: [{X_train.min():.4f}, {X_train.max():.4f}]')

In [ ]:
# ---- 数据准备 ----
# CNN+Attention: 单帧输入 (B, 21)
# 取窗口中间帧作为代表帧
mid_frame = WINDOW_SIZE // 2
X_train_cnn = X_train[:, mid_frame, :]  # (N, 21)
X_val_cnn = X_val[:, mid_frame, :]

# MS-TCN: 窗口输入 (B, T, 21)
X_train_tcn = X_train  # (N, 30, 21)
X_val_tcn = X_val

# 创建 PyTorch DataLoader
BATCH_SIZE = 64

train_ds_cnn = TensorDataset(
    torch.FloatTensor(X_train_cnn),
    torch.LongTensor(y_train),
)
val_ds_cnn = TensorDataset(
    torch.FloatTensor(X_val_cnn),
    torch.LongTensor(y_val),
)

train_ds_tcn = TensorDataset(
    torch.FloatTensor(X_train_tcn),
    torch.LongTensor(y_train),
)
val_ds_tcn = TensorDataset(
    torch.FloatTensor(X_val_tcn),
    torch.LongTensor(y_val),
)

train_loader_cnn = DataLoader(train_ds_cnn, batch_size=BATCH_SIZE, shuffle=True)
val_loader_cnn = DataLoader(val_ds_cnn, batch_size=BATCH_SIZE)
train_loader_tcn = DataLoader(train_ds_tcn, batch_size=BATCH_SIZE, shuffle=True)
val_loader_tcn = DataLoader(val_ds_tcn, batch_size=BATCH_SIZE)

print(f'CNN train batches: {len(train_loader_cnn)}, val: {len(val_loader_cnn)}')
print(f'TCN train batches: {len(train_loader_tcn)}, val: {len(val_loader_tcn)}')

## 2. 1D-CNN + Temporal Attention 模型

### 2.1 架构概览

```
Input (B, 21)
    │
    ▼ unsqueeze(1)
(B, 1, 21)
    │
    ▼ Conv1d(1→32, k=3, pad=1) + BN + ReLU
(B, 32, 21)
    │
    ▼ Conv1d(32→64, k=3, pad=1) + BN + ReLU + MaxPool(2)
(B, 64, 10)
    │
    ▼ Conv1d(64→128, k=3, pad=1) + BN + ReLU + MaxPool(2)
(B, 128, 5)
    │
    ▼ AdaptiveAvgPool1d(1) → squeeze
(B, 128)
    │
    ▼ TemporalAttention(128)
(B, 128)
    │
    ▼ Linear(128 → num_classes)
(B, 46) logits
```

### 2.2 Temporal Attention 机制

使用 **加性注意力 (Bahdanau-style)**：

$$
\mathbf{e} = \mathbf{W}_e \cdot \tanh(\mathbf{W}_h \mathbf{h} + \mathbf{v})
$$

$$
\alpha = \sigma(\mathbf{e})
$$

$$
\tilde{\mathbf{h}} = \alpha \odot \mathbf{h}
$$

其中：
- $\mathbf{h} \in \mathbb{R}^{C}$: 通道特征向量
- $\mathbf{W}_h \in \mathbb{R}^{C \times C}$: 变换矩阵
- $\mathbf{W}_e \in \mathbb{R}^{1 \times C}$: 投影矩阵
- $\mathbf{v} \in \mathbb{R}^{C}$: 可学习上下文向量
- $\alpha$: 逐通道注意力权重

In [ ]:
# ============================================================
# L1 CNN+Attention 模型定义
# 对应 glove_relay/src/models/l1_cnn_attention.py
# ============================================================

class TemporalAttention(nn.Module):
    """加性注意力 (Bahdanau-style)。"""
    
    def __init__(self, in_channels: int):
        super().__init__()
        self.W_h = nn.Linear(in_channels, in_channels, bias=False)
        self.W_e = nn.Linear(in_channels, 1, bias=False)
        self.v = nn.Parameter(torch.randn(in_channels))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, C)
        returns: (B, C)
        """
        v = self.v.unsqueeze(0).expand(x.size(0), -1)  # (B, C)
        energy = torch.tanh(self.W_h(x) + v)            # (B, C)
        energy = self.W_e(energy)                         # (B, 1)
        alpha = torch.softmax(energy, dim=1)              # (B, 1)
        return alpha.squeeze(-1) * x                      # (B, C)


class L1CNNAttention(nn.Module):
    """
    1D-CNN + Temporal Attention (≈34K params)。
    输入: (B, 21) 单帧
    输出: (B, num_classes) logits
    """
    
    def __init__(self, input_dim: int = 21, num_classes: int = 20):
        super().__init__()
        self.input_dim = input_dim
        self.num_classes = num_classes
        
        # Conv backbone
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        
        # Attention + classifier
        self.attention = TemporalAttention(128)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(128, num_classes)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (B, 21) → (B, 1, 21)
        if x.dim() == 2:
            x = x.unsqueeze(1)
        
        x = F.relu(self.bn1(self.conv1(x)))     # (B, 32, 21)
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))  # (B, 64, 10)
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))  # (B, 128, 5)
        
        x = self.gap(x).squeeze(-1)              # (B, 128)
        x = self.attention(x)                     # (B, 128)
        return self.fc(x)                         # (B, num_classes)
    
    def count_params(self) -> int:
        return sum(p.numel() for p in self.parameters())


# 实例化并验证
model_cnn = L1CNNAttention(input_dim=FEATURE_COUNT, num_classes=NUM_CLASSES)
print(f'CNN+Attention parameters: {model_cnn.count_params():,}')

# 验证输入输出形状
dummy_in = torch.randn(4, FEATURE_COUNT)
dummy_out = model_cnn(dummy_in)
print(f'Input: {dummy_in.shape} → Output: {dummy_out.shape}')

In [ ]:
# ---- 层级输出可视化 ----
model_cnn.eval()
x = torch.randn(1, FEATURE_COUNT)

print('CNN+Attention Layer-by-Layer Shape Trace:')
print('-' * 50)

h = x.unsqueeze(1)
print(f'Input:       {x.shape}')
print(f'Unsqueeze:   {h.shape}')

h = F.relu(model_cnn.bn1(model_cnn.conv1(h)))
print(f'Conv1+BN+ReLU: {h.shape}')

h = model_cnn.pool2(F.relu(model_cnn.bn2(model_cnn.conv2(h))))
print(f'Conv2+Pool:   {h.shape}')

h = model_cnn.pool3(F.relu(model_cnn.bn3(model_cnn.conv3(h))))
print(f'Conv3+Pool:   {h.shape}')

h = model_cnn.gap(h).squeeze(-1)
print(f'GAP:          {h.shape}')

h = model_cnn.attention(h)
print(f'Attention:    {h.shape}')

h = model_cnn.fc(h)
print(f'FC output:    {h.shape}')

## 3. Multi-Scale TCN 模型

### 3.1 架构概览

```
Input (B, T, 21)
    │
    ▼ permute → (B, 21, T)
    ▼ Conv1d(21→32, k=1)   // 通道投影
(B, 32, T)
    │
    ▼ ResidualBlock(dilation=1)
(B, 32, T)
    │
    ▼ ResidualBlock(dilation=2)
(B, 64, T)
    │
    ▼ ResidualBlock(dilation=4)
(B, 64, T)
    │
    ▼ AdaptiveAvgPool1d(1) → squeeze
(B, 64)
    │
    ▼ Dropout(0.3) → Linear(64 → num_classes)
(B, 46) logits
```

### 3.2 膨胀卷积（Dilated Convolution）

膨胀卷积在不增加参数的情况下扩大感受野：

$$
\text{RF} = 1 + (k-1) \times d
$$

其中 $k$ 是卷积核大小，$d$ 是膨胀率。

| Stage | Dilation | Kernel | 感受野 |
|-------|----------|--------|--------|
| 1 | 1 | 3 | 3 frames (30ms) |
| 2 | 2 | 3 | 7 frames (70ms) |
| 3 | 4 | 3 | 15 frames (150ms) |

### 3.3 残差块（Residual Block）

$$
\mathbf{y} = \text{ReLU}(\text{BN}(\text{Conv1d}(\mathbf{x}))) + \mathbf{x}
$$

当输入输出通道数不同时，使用 1×1 卷积投影 shortcut：

$$
\mathbf{y} = \text{ReLU}(\text{BN}(\text{Conv1d}(\mathbf{x}))) + \text{Conv1d}_{1 \times 1}(\mathbf{x})
$$

In [ ]:
# ============================================================
# L1 MS-TCN 模型定义
# 对应 glove_relay/src/models/l1_ms_tcn.py
# ============================================================

class ResidualBlock(nn.Module):
    """膨胀卷积残差块。"""
    
    def __init__(self, in_ch: int, out_ch: int, dilation: int, kernel_size: int = 3):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.bn = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = self.relu(self.bn(self.conv(x)))
        if self.downsample is not None:
            identity = self.downsample(identity)
        return out + identity


class L1MSTCN(nn.Module):
    """
    Multi-Scale TCN (≈12K params)。
    输入: (B, T, 21) 窗口
    输出: (B, num_classes) logits
    """
    
    def __init__(self, input_dim: int = 21, num_classes: int = 20,
                 base_channels: int = 32, num_stages: int = 3):
        super().__init__()
        self.input_dim = input_dim
        self.num_classes = num_classes
        
        # 通道投影
        self.input_proj = nn.Conv1d(input_dim, base_channels, 1)
        
        # 膨胀残差阶段
        self.stages = nn.ModuleList()
        in_ch = base_channels
        for i in range(num_stages):
            dilation = 2 ** i  # 1, 2, 4
            out_ch = base_channels * 2 if i >= 1 else base_channels
            self.stages.append(ResidualBlock(in_ch, out_ch, dilation))
            in_ch = out_ch
        
        # 分类头
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(in_ch, num_classes)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (B, T, 21) → (B, 21, T)
        if x.dim() == 2:
            x = x.unsqueeze(1)
        x = x.permute(0, 2, 1)
        
        x = self.input_proj(x)    # (B, 32, T)
        for stage in self.stages:
            x = stage(x)
        
        x = self.gap(x).squeeze(-1)  # (B, C)
        x = self.dropout(x)
        return self.fc(x)
    
    def count_params(self) -> int:
        return sum(p.numel() for p in self.parameters())


model_tcn = L1MSTCN(input_dim=FEATURE_COUNT, num_classes=NUM_CLASSES)
print(f'MS-TCN parameters: {model_tcn.count_params():,}')

dummy_in = torch.randn(4, WINDOW_SIZE, FEATURE_COUNT)
dummy_out = model_tcn(dummy_in)
print(f'Input: {dummy_in.shape} → Output: {dummy_out.shape}')

## 4. 训练循环

### 4.1 损失函数

**交叉熵损失（Cross-Entropy Loss）**：
$$
\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \sum_{c=1}^{C} y_{i,c} \log \hat{y}_{i,c}
$$

其中 $y_{i,c}$ 是 one-hot 标签，$\hat{y}_{i,c} = \text{softmax}(\mathbf{z}_i)_c$。

### 4.2 优化器

**AdamW**（带权重衰减的 Adam）：

$$
\theta_{t+1} = \theta_t - \eta \left( \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} + \lambda \theta_t \right)
$$

### 4.3 学习率调度

- CNN+Attention: **StepLR** (step=20, gamma=0.5)
- MS-TCN: **CosineAnnealingLR** ($T_{\max}$ = epochs)

In [ ]:
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = 80,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    scheduler_type: str = 'step',
    step_size: int = 20,
    gamma: float = 0.5,
    patience: int = 15,
    model_name: str = 'model',
) -> Dict[str, List[float]]:
    """
    通用训练循环。
    
    Parameters
    ----------
    model : nn.Module
        要训练的模型
    train_loader, val_loader : DataLoader
        训练/验证数据加载器
    epochs : int
        最大训练轮数
    lr : float
        初始学习率
    weight_decay : float
        AdamW 权重衰减
    scheduler_type : str
        'step' 或 'cosine'
    patience : int
        早停耐心值
    model_name : str
        模型名称（用于日志）
        
    Returns
    -------
    history : dict
        训练历史
    """
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    if scheduler_type == 'step':
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)
    else:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'lr': [],
    }
    
    best_val_acc = 0.0
    best_epoch = 0
    no_improve = 0
    
    for epoch in range(1, epochs + 1):
        # ---- Training ----
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * xb.size(0)
            train_correct += (logits.argmax(-1) == yb).sum().item()
            train_total += xb.size(0)
        
        scheduler.step()
        
        # ---- Validation ----
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
                val_correct += (logits.argmax(-1) == yb).sum().item()
                val_total += xb.size(0)
        
        # Record
        tl = train_loss / max(train_total, 1)
        ta = train_correct / max(train_total, 1)
        vl = val_loss / max(val_total, 1)
        va = val_correct / max(val_total, 1)
        
        history['train_loss'].append(tl)
        history['val_loss'].append(vl)
        history['train_acc'].append(ta)
        history['val_acc'].append(va)
        history['lr'].append(scheduler.get_last_lr()[0])
        
        # Best model
        if va > best_val_acc:
            best_val_acc = va
            best_epoch = epoch
            no_improve = 0
            # Save best
            save_dir = PROJECT_ROOT / 'checkpoints'
            save_dir.mkdir(exist_ok=True)
            torch.save(model.state_dict(), save_dir / f'{model_name}_best.pt')
            marker = ' ★'
        else:
            no_improve += 1
            marker = ''
        
        if epoch % 10 == 0 or marker:
            print(f'  Epoch {epoch:3d}/{epochs} | '
                  f'Train Loss: {tl:.4f}  Acc: {ta:.4f} | '
                  f'Val Loss: {vl:.4f}  Acc: {va:.4f} | '
                  f'LR: {scheduler.get_last_lr()[0]:.6f}{marker}')
        
        # Early stopping
        if no_improve >= patience:
            print(f'  Early stopping at epoch {epoch} (no improvement for {patience} epochs)')
            break
    
    print(f'\n  Best val accuracy: {best_val_acc:.4f} at epoch {best_epoch}')
    history['best_val_acc'] = best_val_acc
    history['best_epoch'] = best_epoch
    return history

In [ ]:
# ---- 训练 CNN+Attention ----
print('=' * 60)
print('Training L1 CNN+Attention Model')
print('=' * 60)

model_cnn = L1CNNAttention(input_dim=FEATURE_COUNT, num_classes=NUM_CLASSES)
history_cnn = train_model(
    model_cnn,
    train_loader_cnn,
    val_loader_cnn,
    epochs=80,
    lr=1e-3,
    scheduler_type='step',
    step_size=20,
    gamma=0.5,
    model_name='l1_cnn_attention',
)

In [ ]:
# ---- 训练 MS-TCN ----
print('=' * 60)
print('Training L1 MS-TCN Model')
print('=' * 60)

model_tcn = L1MSTCN(input_dim=FEATURE_COUNT, num_classes=NUM_CLASSES)
history_tcn = train_model(
    model_tcn,
    train_loader_tcn,
    val_loader_tcn,
    epochs=80,
    lr=1e-3,
    scheduler_type='cosine',
    model_name='l1_ms_tcn',
)

In [ ]:
# ---- 训练曲线对比 ----
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric, title in zip(
    axes,
    ['train_loss', 'val_loss', 'val_acc'],
    ['Training Loss', 'Validation Loss', 'Validation Accuracy'],
):
    ax.plot(history_cnn[metric], label='CNN+Attention', color='#e74c3c')
    ax.plot(history_tcn[metric], label='MS-TCN', color='#3498db')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('L1 Model Training Comparison', fontsize=13)
plt.tight_layout()
plt.show()

print(f'\nCNN+Attention best val acc: {history_cnn["best_val_acc"]:.4f} (epoch {history_cnn["best_epoch"]})')
print(f'MS-TCN best val acc:       {history_tcn["best_val_acc"]:.4f} (epoch {history_tcn["best_epoch"]})')

## 5. 模型评估

### 5.1 混淆矩阵

混淆矩阵 $\mathbf{M} \in \mathbb{R}^{C \times C}$，其中 $M_{ij}$ 表示真实类别为 $i$ 被预测为 $j$ 的样本数。

### 5.2 评估指标

**逐类精度 (Per-class Accuracy)**:

$$
\text{Acc}_c = \frac{M_{cc}}{\sum_j M_{cj}}
$$

**宏平均 F1 (Macro F1)**:

$$
F1_{\text{macro}} = \frac{1}{C} \sum_{c=1}^{C} \frac{2 \cdot P_c \cdot R_c}{P_c + R_c}
$$

其中 $P_c = M_{cc} / \sum_i M_{ic}$，$R_c = M_{cc} / \sum_j M_{cj}$

In [ ]:
def evaluate_model(model: nn.Module, loader: DataLoader, num_classes: int) -> Dict:
    """全面评估模型。"""
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            probs = F.softmax(logits, dim=-1)
            preds = logits.argmax(-1).cpu()
            all_preds.append(preds)
            all_labels.append(yb)
            all_probs.append(probs.cpu())
    
    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    all_probs = torch.cat(all_probs)
    
    # 混淆矩阵
    cm = torch.zeros(num_classes, num_classes, dtype=torch.long)
    for t, p in zip(all_labels, all_preds):
        cm[t, p] += 1
    
    # 逐类精度
    per_class_acc = cm.diag() / cm.sum(dim=1).clamp(min=1)
    
    # Precision, Recall, F1
    precision = cm.diag() / cm.sum(dim=0).clamp(min=1)
    recall = cm.diag() / cm.sum(dim=1).clamp(min=1)
    f1 = 2 * precision * recall / (precision + recall).clamp(min=1e-8)
    
    # Top-k accuracy
    top3_correct = (all_probs.topk(3, dim=1).indices == all_labels.unsqueeze(1)).any(dim=1).float().mean()
    
    overall_acc = (all_preds == all_labels).float().mean()
    
    return {
        'accuracy': overall_acc.item(),
        'top3_accuracy': top3_correct.item(),
        'macro_f1': f1.mean().item(),
        'confusion_matrix': cm.numpy(),
        'per_class_acc': per_class_acc.numpy(),
        'per_class_f1': f1.numpy(),
        'predictions': all_preds.numpy(),
        'labels': all_labels.numpy(),
    }


# ---- 评估两个模型 ----
model_cnn = model_cnn.to(DEVICE)
model_tcn = model_tcn.to(DEVICE)

eval_cnn = evaluate_model(model_cnn, val_loader_cnn, NUM_CLASSES)
eval_tcn = evaluate_model(model_tcn, val_loader_tcn, NUM_CLASSES)

print('CNN+Attention:')
print(f'  Accuracy:    {eval_cnn["accuracy"]:.4f}')
print(f'  Top-3 Acc:   {eval_cnn["top3_accuracy"]:.4f}')
print(f'  Macro F1:    {eval_cnn["macro_f1"]:.4f}')
print()
print('MS-TCN:')
print(f'  Accuracy:    {eval_tcn["accuracy"]:.4f}')
print(f'  Top-3 Acc:   {eval_tcn["top3_accuracy"]:.4f}')
print(f'  Macro F1:    {eval_tcn["macro_f1"]:.4f}')

In [ ]:
# ---- 混淆矩阵可视化 ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for ax, eval_result, title in [
    (ax1, eval_cnn, 'CNN+Attention'),
    (ax2, eval_tcn, 'MS-TCN'),
]:
    cm = eval_result['confusion_matrix'].astype(float)
    # 归一化
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'{title} Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()

# ---- 逐类精度对比 ----
fig, ax = plt.subplots(figsize=(14, 5))
x_pos = np.arange(NUM_CLASSES)
width = 0.35

ax.bar(x_pos - width/2, eval_cnn['per_class_acc'], width, label='CNN+Attention', color='#e74c3c', alpha=0.8)
ax.bar(x_pos + width/2, eval_tcn['per_class_acc'], width, label='MS-TCN', color='#3498db', alpha=0.8)
ax.set_xlabel('Gesture Class ID')
ax.set_ylabel('Accuracy')
ax.set_title('Per-Class Accuracy')
ax.set_xticks(x_pos)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. QAT 量化感知训练 → TFLite INT8 导出

### 6.1 量化原理

**INT8 对称量化**公式：

$$
q = \text{round}\left(\frac{x}{s}\right), \quad s = \frac{\max(|x|)}{127}
$$

**反量化**：

$$
\hat{x} = q \cdot s
$$

### 6.2 QAT 流程

1. **准备**: 在模型中插入伪量化节点 (Fake Quantize)
2. **训练**: 正常训练，但梯度通过 Straight-Through Estimator 传递
3. **转换**: 将浮点权重转换为 INT8
4. **校准**: 用代表性数据确定激活的量化范围

### 6.3 模型大小对比

| 格式 | 大小 (est.) | 推理速度 |
|------|------------|----------|
| FP32 PyTorch | ~136 KB | ~2 ms (CPU) |
| INT8 TFLite | ~34 KB | ~0.5 ms (ESP32-S3) |

In [ ]:
def export_tflite_int8(model: nn.Module, sample_input: torch.Tensor,
                       model_name: str = 'l1_model') -> str:
    """
    导出 TFLite INT8 模型。
    
    Parameters
    ----------
    model : nn.Module
        训练好的 PyTorch 模型
    sample_input : torch.Tensor
        代表性输入（用于量化校准）
    model_name : str
        输出文件名前缀
        
    Returns
    -------
    str
        TFLite 文件路径
    """
    model.eval().cpu()
    
    # 方法 1: 通过 ONNX 中转
    try:
        import onnx
        from onnx_tf.backend import prepare
        import tensorflow as tf
        has_onnx = True
    except ImportError:
        has_onnx = False
    
    # 方法 2: 直接使用 PyTorch quantization
    # Dynamic quantization (最简单的路径)
    quantized_model = torch.quantization.quantize_dynamic(
        model,
        {nn.Linear},  # 量化 Linear 层
        dtype=torch.qint8,
    )
    
    # 保存量化后的 PyTorch 模型
    save_dir = PROJECT_ROOT / 'checkpoints'
    save_dir.mkdir(exist_ok=True)
    
    # 保存为 TorchScript
    scripted = torch.jit.trace(model, sample_input)
    torchscript_path = save_dir / f'{model_name}_fp32.pt'
    scripted.save(str(torchscript_path))
    
    # 模型大小
    fp32_size = torchscript_path.stat().st_size
    
    print(f'Saved TorchScript model: {torchscript_path}')
    print(f'  FP32 size: {fp32_size / 1024:.1f} KB')
    
    # 推理延迟基准
    model.eval()
    with torch.no_grad():
        # Warmup
        for _ in range(100):
            _ = model(sample_input)
        # Benchmark
        start = time.perf_counter()
        N = 1000
        for _ in range(N):
            _ = model(sample_input)
        elapsed = (time.perf_counter() - start) / N * 1000  # ms
    
    print(f'  Inference latency: {elapsed:.3f} ms/sample (CPU, PyTorch)')
    print(f'  Throughput: {1000/elapsed:.0f} samples/sec')
    
    return str(torchscript_path)


# ---- 导出 CNN+Attention ----
print('Exporting CNN+Attention...')
sample_cnn = torch.randn(1, FEATURE_COUNT)
path_cnn = export_tflite_int8(model_cnn, sample_cnn, 'l1_cnn_attention')

print()

# ---- 导出 MS-TCN ----
print('Exporting MS-TCN...')
sample_tcn = torch.randn(1, WINDOW_SIZE, FEATURE_COUNT)
path_tcn = export_tflite_int8(model_tcn, sample_tcn, 'l1_ms_tcn')

## 7. 模型大小与推理延迟基准

### 7.1 ESP32-S3 性能预估

| 指标 | CNN+Attention | MS-TCN |
|------|--------------|--------|
| 参数量 | ~34K | ~12K |
| FP32 模型大小 | ~136 KB | ~48 KB |
| INT8 模型大小 | ~34 KB | ~12 KB |
| ESP32-S3 推理 (est.) | ~2 ms | ~1 ms |
| TFLite Arena | ~30 KB | ~20 KB |

### 7.2 内存预算

ESP32-S3 N16R8 内存：
- Flash: 16 MB（代码 + 模型权重）
- PSRAM: 8 MB（滑动窗口 + TFLite Arena）
- SRAM: 512 KB（FreeRTOS 任务栈 + 变量）

```
模型权重:    34 KB  (INT8, Flash)
TFLite Arena: 30 KB  (PSRAM)
滑动窗口:     2.5 KB (PSRAM)
FreeRTOS 栈:  8 KB × 3 tasks = 24 KB (SRAM)
───────────────────────────────────────
总计:         ~90 KB  (远低于内存上限)
```

In [ ]:
# ---- 参数量与大小分析 ----
def analyze_model(model: nn.Module, name: str, input_shape: tuple):
    """分析模型参数量和大小。"""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # 按层类型分组
    layer_counts = {}
    for name_, module in model.named_modules():
        type_name = type(module).__name__
        if type_name not in layer_counts:
            layer_counts[type_name] = {'count': 0, 'params': 0}
        layer_counts[type_name]['count'] += 1
        layer_counts[type_name]['params'] += sum(p.numel() for p in module.parameters(recurse=False))
    
    print(f'\n{'='*50}')
    print(f'Model: {name}')
    print(f'Input shape: {input_shape}')
    print(f'Total params: {total_params:,}')
    print(f'Trainable params: {trainable_params:,}')
    print(f'FP32 size: ~{total_params * 4 / 1024:.1f} KB')
    print(f'INT8 size: ~{total_params / 1024:.1f} KB')
    print(f'\nLayer breakdown:')
    for lt, info in sorted(layer_counts.items(), key=lambda x: -x[1]['params']):
        if info['params'] > 0:
            print(f'  {lt:25s}: {info["count"]:3d} layers, {info["params"]:>8,} params')


analyze_model(model_cnn, 'L1 CNN+Attention', '(B, 21)')
analyze_model(model_tcn, 'L1 MS-TCN', '(B, 30, 21)')

## 8. ESP32-S3 部署 — C 头文件生成

将模型权重导出为 C 语言头文件，用于 ESP32-S3 固件编译。

### 8.1 TFLite Micro 部署路径

```
PyTorch model → TorchScript → TFLite → xxd → C header → ESP32 firmware
```

### 8.2 生成命令

```bash
# TorchScript → TFLite
python -c "
import torch
model = torch.jit.load('checkpoints/l1_cnn_attention_fp32.pt')
sm = torch.jit.trace(model, torch.randn(1, 21))
# ... (需要 ONNX → TF → TFLite 路径)
"

# TFLite → C header
xxd -i model.tflite > model_data.h
```

In [ ]:
def generate_c_header(state_dict: dict, model_name: str = 'l1_cnn_attention') -> str:
    """
    将 PyTorch state_dict 转换为 C 头文件格式。
    注：这是一个辅助函数，实际部署推荐使用 TFLite Micro 路径。
    
    Parameters
    ----------
    state_dict : dict
        PyTorch 模型的 state_dict
    model_name : str
        模型名称
        
    Returns
    -------
    str
        C 头文件内容
    """
    lines = []
    lines.append(f'// Auto-generated model weights for {model_name}')
    lines.append(f'// Generated by EchoGlove training pipeline')
    lines.append(f'#ifndef {model_name.upper()}_WEIGHTS_H')
    lines.append(f'#define {model_name.upper()}_WEIGHTS_H')
    lines.append('')
    lines.append('#include <cstdint>')
    lines.append('')
    
    total_bytes = 0
    for param_name, tensor in state_dict.items():
        # 清理名称
        clean_name = param_name.replace('.', '_')
        flat = tensor.cpu().numpy().flatten()
        
        # INT8 量化
        scale = np.max(np.abs(flat)) / 127.0 if np.max(np.abs(flat)) > 0 else 1.0
        quantized = np.round(flat / scale).astype(np.int8)
        
        lines.append(f'// {param_name}: shape={list(tensor.shape)}, scale={scale:.6f}')
        lines.append(f'const float {model_name}_{clean_name}_scale = {scale:.6f}f;')
        lines.append(f'const int8_t {model_name}_{clean_name}[] = {{')
        
        # 每行 16 个值
        for i in range(0, len(quantized), 16):
            chunk = quantized[i:i+16]
            values = ', '.join(f'{v:4d}' for v in chunk)
            lines.append(f'  {values},')
        
        lines.append('};')
        lines.append(f'const size_t {model_name}_{clean_name}_len = {len(quantized)};')
        lines.append('')
        total_bytes += len(quantized)
    
    lines.append(f'// Total weight bytes (INT8): {total_bytes}')
    lines.append(f'#endif // {model_name.upper()}_WEIGHTS_H')
    
    return '\n'.join(lines)


# ---- 生成 C 头文件 ----
model_cnn.eval()
c_header = generate_c_header(model_cnn.state_dict(), 'l1_cnn_attention')

save_dir = PROJECT_ROOT / 'glove_firmware' / 'lib' / 'Models'
save_dir.mkdir(parents=True, exist_ok=True)
header_path = save_dir / 'l1_model_weights.h'

with open(header_path, 'w') as f:
    f.write(c_header)

print(f'Generated C header: {header_path}')
print(f'File size: {header_path.stat().st_size / 1024:.1f} KB')
print(f'\nFirst 30 lines:')
with open(header_path) as f:
    for i, line in enumerate(f):
        if i >= 30: break
        print(line, end='')

## 9. 模型注册表与热切换

### 9.1 YAML 配置

模型通过 `model_config.yaml` 进行热切换：

```yaml
active_model: cnn_attention_v2
models:
  cnn_attention_v2:
    class: L1EdgeModel
    path: checkpoints/l1_cnn_attention_best.pt
    input_dim: 21
    num_classes: 46
  ms_tcn_v1:
    class: MSTCNModel
    path: checkpoints/l1_ms_tcn_best.pt
    input_dim: 21
    num_classes: 46
```

### 9.2 BaseModel 接口

所有模型实现统一接口：

```python
class BaseModel(ABC):
    def predict(self, x: Tensor) -> tuple[int, float]: ...
    def get_config(self) -> dict: ...
    def get_model_info(self) -> dict: ...
```

In [ ]:
# ---- 模型注册表示例 ----
model_registry = {
    'cnn_attention_v2': {
        'model_class': 'L1CNNAttention',
        'input_type': 'single_frame',  # (B, 21)
        'params': model_cnn.count_params(),
        'val_acc': eval_cnn['accuracy'],
    },
    'ms_tcn_v1': {
        'model_class': 'L1MSTCN',
        'input_type': 'window',  # (B, T, 21)
        'params': model_tcn.count_params(),
        'val_acc': eval_tcn['accuracy'],
    },
}

print('Model Registry:')
print('-' * 60)
for name, info in model_registry.items():
    print(f'  {name}:')
    print(f'    Class:     {info["model_class"]}')
    print(f'    Input:     {info["input_type"]}')
    print(f'    Params:    {info["params"]:,}')
    print(f'    Val Acc:   {info["val_acc"]:.4f}')

---

## 小结

| 项目 | CNN+Attention | MS-TCN |
|------|--------------|--------|
| 参数量 | ~34K | ~12K |
| 输入 | 单帧 (B, 21) | 窗口 (B, 30, 21) |
| 特点 | 逐帧推理，低延迟 | 时序建模，高精度 |
| 适用 | L1 实时推理 | L1 时序推理 |
| INT8 大小 | ~34 KB | ~12 KB |

**下一步**: 运行 `03_l2_stgcn_deployment.ipynb` 进行 L2 ST-GCN 模型训练与 Relay 部署。